In [1]:
!pip install fastapi uvicorn pyngrok transformers accelerate bitsandbytes nest_asyncio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 45.5 MB/s eta 0:00:00


In [2]:
from huggingface_hub import login

# You will be prompted to paste your HF token (make a token at: https://huggingface.co/settings/tokens)
login()


In [3]:
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json
import re
from typing import List, Dict, Any

# --------- MODEL LOADING ----------
MODEL_NAME = "unsloth/llama-3-8b-Instruct-bnb-4bit"

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()

# --------- NER LABEL SET (11) ----------
LABELS = [
    "COMPONENT",
    "SOFTWARE",
    "ACTION",
    "INPUT",
    "DATA_OBJECT",
    "VALUE",
    "UNIT",
    "STANDARD",
    "FIGURE_REF",
    "CLAIM_ELEMENT",
    "UNKNOWN_ENTITY",
]

# --------- FASTAPI APP ----------
app = FastAPI()


class PredictRequest(BaseModel):
    data: List[Dict[str, Any]]  # [{"text": "..."}]


def _extract_json_block(s: str) -> str:
    """
    Try to extract the first JSON array/object from a model output.
    We expect a JSON array: [{"start":..,"end":..,"text":"..","label":".."}, ...]
    """
    s = s.strip()
    # Common: model echoes prompt; grab first [...] block
    m = re.search(r"\[\s*{.*}\s*\]", s, flags=re.DOTALL)
    if m:
        return m.group(0)
    # Fallback: try {...} (if it returns an object)
    m = re.search(r"\{\s*\"spans\"\s*:\s*\[.*\]\s*\}", s, flags=re.DOTALL)
    if m:
        return m.group(0)
    return ""


def _find_all_occurrences(text: str, sub: str):
    """Yield all (start,end) for exact substring matches."""
    if not sub:
        return
    start = 0
    while True:
        i = text.find(sub, start)
        if i == -1:
            break
        yield i, i + len(sub)
        start = i + 1


def ner_spans(text: str) -> List[Dict[str, Any]]:
    """
    Prompt-based span extraction.
    Returns list of dicts: {"start": int, "end": int, "text": str, "label": str}
    where start/end refer to indices in the ORIGINAL input text (0-based, end-exclusive).
    """
    prompt = f"""
You are a patent NER span extractor for Label Studio.

TASK:
Extract ONLY meaningful entity-like spans from the given text using the label set below.
Return JSON ONLY.

LABEL SET (choose exactly one per span):
{", ".join(LABELS)}

RULES (critical):
- Use exact substrings from the input text (copy-paste exact characters).
- Prefer longer meaningful noun phrases over single tokens.
- Do NOT tag generic boilerplate nouns like "system", "method", "embodiment" unless they are clearly a meaningful entity in context.
- Do NOT tag numbering-only tokens.
- Avoid overlaps unless unavoidable.
- Use UNKNOWN_ENTITY only when it is clearly entity-like but does not fit other labels.
- FIGURE_REF for "FIG. 1", "Fig. 2", "as shown in FIG. 3", etc.
- STANDARD only for externally governed standards/regulations/specs (ISO, IEEE, RFC, GDPR, FDA, CE, etc.).
- CLAIM_ELEMENT for claim-structure words like "wherein", "comprising", "means for" (optional; only if clearly present).

OUTPUT FORMAT:
Return a JSON array of objects, each with:
- "text": exact substring
- "label": one of the labels above

Do NOT include start/end in the JSON.
Do NOT include any commentary.

INPUT TEXT:
\"\"\"{text}\"\"\"
""".strip()

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=800,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False,
        )

    decoded = tokenizer.decode(output[0], skip_special_tokens=True).strip()

    json_str = _extract_json_block(decoded)
    if not json_str:
        return []

    # Parse model JSON
    try:
        parsed = json.loads(json_str)
    except Exception:
        return []

    # Accept either [{"text","label"}...] or {"spans":[...]}
    if isinstance(parsed, dict) and "spans" in parsed:
        spans = parsed["spans"]
    elif isinstance(parsed, list):
        spans = parsed
    else:
        return []

    # Convert to Label Studio style spans with offsets by exact substring matching.
    results = []
    used_ranges = []  # (start,end) to reduce overlaps

    for s in spans:
        if not isinstance(s, dict):
            continue
        span_text = (s.get("text") or "").strip()
        label = (s.get("label") or "").strip()

        if not span_text or label not in LABELS:
            continue

        # Find best occurrence: first non-overlapping match
        chosen = None
        for st, en in _find_all_occurrences(text, span_text):
            overlaps = any(not (en <= a or st >= b) for a, b in used_ranges)
            if not overlaps:
                chosen = (st, en)
                break

        # If everything overlaps, still allow the first occurrence (Label Studio can store overlaps,
        # but we try to avoid them)
        if chosen is None:
            occ = next(_find_all_occurrences(text, span_text), None)
            if occ is None:
                continue
            chosen = occ

        st, en = chosen
        used_ranges.append((st, en))

        results.append(
            {
                "start": st,
                "end": en,
                "text": text[st:en],  # exact
                "label": label,
            }
        )

    return results


@app.get("/health")
def health():
    return {"status": "ok"}


# Label Studio "ML backend" expects /setup
@app.api_route("/setup", methods=["GET", "POST"])
def setup():
    return {
        "from_name": "ner",     # must match your LS config
        "to_name": "text",      # must match your LS config
        "type": "labels",       # span NER
        "labels": LABELS,
    }


@app.post("/predict")
def predict(request: PredictRequest):
    predictions = []

    for item in request.data:
        text = item.get("text", "")
        spans = ner_spans(text)

        # Convert to Label Studio result format
        ls_results = []
        for sp in spans:
            ls_results.append(
                {
                    "from_name": "ner",   # must match LS config
                    "to_name": "text",    # must match LS config
                    "type": "labels",
                    "value": {
                        "start": sp["start"],
                        "end": sp["end"],
                        "text": sp["text"],
                        "labels": [sp["label"]],
                    },
                }
            )

        predictions.append(
            {
                "result": ls_results,
                "score": 1.0,
                "model_version": MODEL_NAME,
            }
        )

    return predictions


Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

In [4]:
import uvicorn, threading, nest_asyncio, time, requests

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# start server in background
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(5)  # give it a few seconds to start

# quick test: local health
print(requests.get("http://127.0.0.1:8000/health").json())


INFO:     Started server process [902]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:44054 - "GET /health HTTP/1.1" 200 OK
{'status': 'ok'}


In [5]:
from google.colab import output

public_url = output.eval_js("google.colab.kernel.proxyPort(8000)")
public_url


'https://8000-gpu-a100-s-2f78f3u247oem-c.asia-southeast1-0.prod.colab.dev'

In [6]:
from pyngrok import ngrok

# kill old tunnels in this session
ngrok.kill()

# your auth token
ngrok.set_auth_token("2PgsprcdKolcczw6ru6HXbLcYfC_7cUSXnTdho7wqZyHYotoF")

# A) random domain
# public_url = ngrok.connect(addr="127.0.0.1:8000")

# B) your reserved free domain
public_url = ngrok.connect(
    addr="127.0.0.1:8000",
    domain="empiristic-mariyah-unprophetically.ngrok-free.dev"
)

print("Public URL:", public_url)


Public URL: NgrokTunnel: "https://empiristic-mariyah-unprophetically.ngrok-free.dev" -> "http://127.0.0.1:8000"


InvalidSchema: No connection adapters were found for '127.0.0.1:8000/api/predictions/bulk/'